# Modelo Preditivo
### Grupo:
* Gabriel Koji Kondo;
* João Vitor Vargas Pereira;
* Rafael Barreto da Cruz;
* Raissa Arruda Casale;
* Ryan Lionel de Souza.

O objetivo deste notebook é prever o próximo título a ser assitido por um usuário da Netflix, baseado em informações sobre o conteúdo consumido pela conta

## 0. Importações

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
import shap

## 1. Carregamento de dados

In [6]:
df_api = pd.read_csv('../dados-transformados/tmdb_completo.csv')
df_viewing = pd.read_csv('../dados-transformados/Viewing_en.csv')
df_viewing[['Title_Name', 'Temporada', 'Episodio']] = df_viewing['Title'].str.split(': ',n=2, expand=True)
df_viewing.drop('Title', axis=1, inplace=True)
df_viewing["Title_Name"] = df_viewing["Title_Name"].str.replace(r"\(.*?\)", "", regex=True)

df = df_viewing.merge(df_api, on="Title_Name", how="left")

## 2. Pré Processamento

In [7]:
# Converter para datetime
df["Start Time"] = pd.to_datetime(df["Start Time"], errors="coerce")

# Ordenar
df = df.sort_values(["Profile Name", "Start Time"])

# Features temporais
df["dia_semana"] = df["Start Time"].dt.dayofweek
df["hora"] = df["Start Time"].dt.hour

# Gênero atual e próximo
df["genero_atual"] = df["genres"]
df["proximo_genero"] = df.groupby("Profile Name")["genres"].shift(-1)

# Remover última linha de cada usuário (não tem próximo)
df = df.dropna(subset=["proximo_genero"])

# Mudança de gênero (CORREÇÃO)
df["mudou_genero"] = (
    df["genero_atual"] != df["proximo_genero"]
).astype(int)

# -------------------------
# DUMMIES (FAZER UMA VEZ SÓ)
# -------------------------

dummies = pd.get_dummies(df["genres"], prefix="flag")

df = pd.concat([df, dummies], axis=1)

# proporção de cada gênero por perfil
props = dummies.groupby(df["Profile Name"]).transform("mean")
props = props.add_prefix("prop_")

df = pd.concat([df, props], axis=1)

## 3. Separação entre treino e teste

## 3.1 Separando os x e y

In [8]:
df["Start Time"] = pd.to_datetime(df["Start Time"], errors="coerce")
df["dia_semana"] = df["Start Time"].dt.dayofweek
df["hora"] = df["Start Time"].dt.hour

genero_dummies = pd.get_dummies(
    df["genero_atual"],
    prefix="gen"
)

df = pd.concat([df, genero_dummies], axis=1)

colunas_remover = [
    "proximo_genero",
    "Start Time",
    "Profile Name",
    "genero_atual"
]

X = df.drop(columns=colunas_remover)
y = df["proximo_genero"]

# X = X.astype(float)

In [9]:
X.select_dtypes(include="object").columns

df["director"] = df["director"].fillna("")

df["director_list"] = df["director"].str.split(", ")

df_dir = df.explode("director_list")

genero_diretor = df_dir[["genres", "director_list"]].drop_duplicates()

matriz = pd.crosstab(
    genero_diretor["genres"],
    genero_diretor["director_list"]
)

similaridade_generos = matriz.dot(matriz.T)

import numpy as np

np.fill_diagonal(similaridade_generos.values, 0)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = DecisionTreeClassifier(max_depth=10)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)

ValueError: could not convert string to float: '00:14:55'